# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col
from pyspark.sql.window import Window

# Reading from Bronze

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

# Data transformations

In [0]:
df.display()
df.printSchema()

Analyzing the table, we can normalize the GEN column so it has only Male and Female, not F and M. Also, we can Rename the column so they are more understandable

# Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Normalizing GEN

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)


# Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Sanity check of dataframe

In [0]:
df.limit(10).display()

# Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

# Sanity check of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers LIMIT 10